# Single and double precision

`options.precision` selects `complex128` (the default) or `complex64`. Single precision halves the memory of a state
and runs faster on wide circuits, at an amplitude error near 10⁻⁷, far below sampling noise.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': '02f7e5c0c4f6:7:0b2c9c',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. The same distribution in both precisions

With the same seed, a 14-qubit random circuit gives almost identical counts in both precisions. Two double-precision runs with different seeds differ far more, from sampling noise alone, so the precision is invisible in the results.

In [2]:
prog = scrambler(14, 8)
c128 = run("quantinuum_helios", prog, shots=200000, seed=1)
c64 = run("quantinuum_helios", prog, shots=200000, seed=1, precision="complex64")
other = run("quantinuum_helios", prog, shots=200000, seed=2)
print("precision reported:", c128["telemetry"]["workload"]["precision"], "/", c64["telemetry"]["workload"]["precision"])
print("Hellinger complex128 vs complex64:     ", round(hellinger(c128["result"]["counts"], c64["result"]["counts"]), 5))
print("Hellinger complex128 vs another seed:  ", round(hellinger(c128["result"]["counts"], other["result"]["counts"]), 5))

precision reported: complex128 / complex64
Hellinger complex128 vs complex64:      0.9999
Hellinger complex128 vs another seed:   0.95736


## 2. Amplitude error

Comparing the returned statevectors directly shows the size of the single-precision rounding error.

In [3]:
prog = scrambler(12, 10)
amp = {}
for precision in ("complex128", "complex64"):
    sv = run("quantinuum_helios", prog, shots=1, precision=precision, return_statevector=True)["result"]["statevector"]
    amp[precision] = np.array([complex(re, im) for re, im in sv])
diff = abs(amp["complex128"] - amp["complex64"])
print(f"largest amplitude difference {diff.max():.2e}; largest probability difference "
      f"{abs(abs(amp['complex128'])**2 - abs(amp['complex64'])**2).max():.2e}")

largest amplitude difference 4.08e-08; largest probability difference 3.11e-09


## 3. Half the memory

The workload report shows the state size halving.

In [4]:
rows = []
for n in (16, 20, 22):
    for precision in ("complex128", "complex64"):
        w = run("quantinuum_helios", ghz(n), shots=16, precision=precision)["telemetry"]["workload"]
        rows.append({"qubits": n, "precision": precision, "state MiB": w["state_bytes"] / 2**20})
pd.DataFrame(rows).pivot(index="qubits", columns="precision", values="state MiB")

precision,complex128,complex64
qubits,,
16,1.0,0.5
20,16.0,8.0
22,64.0,32.0


## 4. Engine time on deep circuits

On deep random circuits single precision is faster because it moves half the bytes. Times are engine-only, from the job's telemetry; they depend on this machine.

In [5]:
rows = []
for n in (16, 18, 20):
    prog = scrambler(n, 10)
    for precision in ("complex128", "complex64"):
        ms = min(run("quantinuum_helios", prog, shots=64, precision=precision)["telemetry"]["breakdown"]["simulation_ms"]
                 for _ in range(3))
        rows.append({"qubits": n, "precision": precision, "engine ms (best of 3)": round(ms, 1)})
t = pd.DataFrame(rows).pivot(index="qubits", columns="precision", values="engine ms (best of 3)")
t["speed-up"] = (t["complex128"] / t["complex64"]).round(2)
t

precision,complex128,complex64,speed-up
qubits,,,
16,14.3,11.5,1.24
18,27.8,21.3,1.31
20,58.2,40.2,1.45


## Takeaways

- complex64 changes amplitudes by about 10⁻⁷, invisible in sampled counts.
- It halves memory and speeds up deep, memory-bound circuits.